In [1]:
import logging
from transformers import logging as tf_logging

tf_logging.set_verbosity_error()

In [2]:
import sys

In [3]:
import asyncio
import json
import os
import time
from pathlib import Path

import dotenv
from tqdm import tqdm

from financial_qa.chunkers.semantic import SemanticChunker
from financial_qa.chunkers.table_split import TableSplitChunker
from financial_qa.embedders import GigaEmbedder
from financial_qa.rag import RAG
from financial_qa.agent.agent_loop import OpenRouterAgentLoop
from financial_qa.agent.gigachat_agent_loop import GigaChatAgentLoop
from financial_qa.evaluation import evaluate_async, load_jsonl

In [4]:
dotenv.load_dotenv('.env')

GIGACHAT_CREDENTIALS = os.getenv('GIGACHAT_CREDENTIALS')
if not GIGACHAT_CREDENTIALS:
    raise ValueError('GIGACHAT_CREDENTIALS is required')

DATASET_FILE = 'dataset.jsonl'
DATASET_SPLIT = None
MAX_QUESTIONS = 50

RAG_DB = 'table_split_semantic_giga_embeddings'
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
MAX_ROWS_PER_CHUNK = 20
TOP_K = 10
EMBED_MODEL = 'EmbeddingsGigaR'
GIGACHAT_SCOPE = 'GIGACHAT_API_PERS'

GEN_MODEL = 'GigaChat-2-Pro'
QUERY_CONCURRENCY = 1

JUDGE_MODEL = 'google/gemini-2.0-flash-lite-001'
JUDGE_PROCESSES = 100
USE_GIGACHAT_JUDGE = True

In [5]:
all_records = load_jsonl(DATASET_FILE)

In [6]:
len(all_records)

449

In [7]:
all_records = load_jsonl(DATASET_FILE)

all_records = [
    all_records[r]
    for r in all_records
    if DATASET_SPLIT is None or all_records[r].get('split') == DATASET_SPLIT
]

seen = set()
records = []
cnt = 0
for r in all_records:
    if r['question_id'] not in seen:
        records.append(r)
        seen.add(r['question_id'])
        cnt += 1
print(cnt)

if MAX_QUESTIONS:
    records = records[:MAX_QUESTIONS]

golden = {r['question_id']: r for r in records}
print(f'Loaded {len(records)} records (split={DATASET_SPLIT!r})')
print('Sample:', json.dumps(records[0], ensure_ascii=False, indent=2))

449
Loaded 50 records (split=None)
Sample: {
  "question_id": "q_00d660efcf3e4607",
  "question": "Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?",
  "split": "test",
  "gold_evidence": [
    {
      "doc_id": "alfa_2025_annual",
      "pages": [
        103
      ]
    }
  ],
  "gold_answer": "1,151 тыс. белорусских рублей"
}


In [8]:
text_chunker = SemanticChunker(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunker = TableSplitChunker(text_chunker=text_chunker, max_rows_per_chunk=MAX_ROWS_PER_CHUNK)
embedder = GigaEmbedder(
    credentials=GIGACHAT_CREDENTIALS,
    model=EMBED_MODEL,
    scope=GIGACHAT_SCOPE,
)
rag = RAG(
    chunker=chunker,
    embedder=embedder,
    data_dir='data/parsed',
    store_dir='indexes',
    name=RAG_DB,
    top_k=TOP_K,
)

store_dir = Path('indexes') / RAG_DB
has_index = store_dir.exists() and any(store_dir.glob('*.npz'))
if not has_index:
    print('No index found; running precalc...')
    rag.precalc()
else:
    print(f'Using existing index at {store_dir}')

loop = GigaChatAgentLoop(
    rag=rag,
    model=GEN_MODEL,
    credentials=GIGACHAT_CREDENTIALS,
    scope=GIGACHAT_SCOPE,
)

Using existing index at indexes/table_split_semantic_giga_embeddings


In [9]:
async def run_queries(records):
    predictions = {}
    errors = []
    timings = []
    semaphore = asyncio.Semaphore(QUERY_CONCURRENCY)

    async def _query_one(rec):
        start = time.perf_counter()
        try:
            answer, confidence = await loop.aquery(rec['question'])
            error = None
        except Exception as e:
            answer = ''
            confidence = None
            error = str(e)
        elapsed = time.perf_counter() - start
        return {
            'question_id': rec['question_id'],
            'question': rec['question'],
            'answer': answer,
            'evidence': [],
            'confidence': confidence,
            'error': error,
            'elapsed_s': elapsed,
        }

    async def _bound(rec):
        async with semaphore:
            return await _query_one(rec)

    tasks = {asyncio.create_task(_bound(rec)): rec for rec in records}
    progress = tqdm(total=len(tasks), desc='Querying agent', unit='question')
    for task in asyncio.as_completed(tasks):
        result = await task
        predictions[result['question_id']] = result
        if result['error']:
            errors.append(result)
        timings.append(result['elapsed_s'])
        progress.update(1)
        progress.set_postfix(
            errors=len(errors),
            avg_s=f"{sum(timings)/len(timings):.2f}",
            last_conf=result['confidence'],
        )
    progress.close()
    return predictions, errors

predicted, query_errors = await run_queries(records)
print(f'Done: {len(predicted)} answers, {len(query_errors)} errors')

Querying agent:   0%|          | 0/50 [00:00<?, ?question/s]

Querying agent:   8%|▊         | 4/50 [00:19<04:03,  5.30s/question, avg_s=4.83, errors=0, last_conf=0.848]

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 12s…


Querying agent:  28%|██▊       | 14/50 [01:24<02:57,  4.93s/question, avg_s=6.06, errors=0, last_conf=0.798]

CancelledError: 

In [ ]:
query_errors

[]

In [ ]:
result = await evaluate_async(
    golden=golden,
    predicted=predicted,
    model=JUDGE_MODEL,
    detailed_result=True,
    include_evidence=False,
    use_processes=True,
    max_workers=JUDGE_PROCESSES,
    progress_desc='LLM-as-judge',
)

LLM-as-judge: 100%|██████████| 50/50 [00:01<00:00, 34.27question/s, accuracy=56.00%, correct=28, errors=0]


In [ ]:
result['correct'] / result['total']

0.56

In [ ]:
for res in result['results'][0:10]:
    print(res)
    print('-' * 75)

{'question_id': 'q_00d660efcf3e4607', 'question': 'Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?', 'gold_answer': '1,151 тыс. белорусских рублей', 'predicted_answer': 'Отчет не содержит информации о общей сумме финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года.', 'judge_score': 0, 'judge_reasoning': 'Предсказанный ответ утверждает, что информация отсутствует, в то время как золотой ответ предоставляет конкретное число.'}
---------------------------------------------------------------------------
{'question_id': 'q_017f831a3f0e16a3', 'question': 'Применяет ли ЗАО «Альфа-Банк» учёт хеджирования в отношении производных финансовых инструментов согласно финансовой отчётности за 2022 год?', 'gold_answer': 'Нет, Банк не применяет учёт хеджирования.', 'predicted_answer': 'Банк не приме

In [ ]:
import shutil
shutil.make_archive("logs", "zip", "logs")

'/Users/kitlix/CProjects/hse/ai360_proj_may_2026/ai360-financial-qa/logs.zip'